### Слияние с CPT

In [ ]:
import torch
import json
import numpy as np
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from typing import Dict, List
import gc
import os
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
import nltk
from nltk.tokenize import word_tokenize
from sentence_transformers import SentenceTransformer

In [ ]:
#необходимые данные для NLTK
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)

In [ ]:
torch.backends.cudnn.enabled = False
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [ ]:
semantic_model = None

def init_semantic_model(): #модель для семантического сходства"
    global semantic_model
    if semantic_model is None:
        print("Загрузка модели для семантического сходства...")
        try:
            semantic_model = SentenceTransformer('all-MiniLM-L6-v2', device="cuda" if torch.cuda.is_available() else "cpu")
        except:
            semantic_model = SentenceTransformer('all-MiniLM-L6-v2', device="cpu")
    return semantic_model

Методы слияния

In [ ]:
def merge_linear(task_vectors: List[Dict], weights: List[float] = None) -> Dict:
    if weights is None:
        weights = [1.0 / len(task_vectors)] * len(task_vectors)

    merged = {}
    for key in task_vectors[0].keys():
        merged[key] = sum(tv[key] * w for tv, w in zip(task_vectors, weights))
    return merged

def merge_ties(task_vectors: List[Dict], top_k: float = 0.2) -> Dict:
    merged = {}
    for key in task_vectors[0].keys():
        vectors = torch.stack([tv[key] for tv in task_vectors])
        magnitudes = torch.abs(vectors)
        threshold = torch.quantile(magnitudes, 1 - top_k, dim=0)
        mask = magnitudes >= threshold
        trimmed = vectors * mask

        sign_sum = torch.sign(trimmed).sum(dim=0)
        elected_sign = torch.sign(sign_sum)

        sign_mask = torch.sign(trimmed) == elected_sign.unsqueeze(0)
        masked_trimmed = trimmed * sign_mask
        merged[key] = masked_trimmed.sum(dim=0) / (sign_mask.sum(dim=0) + 1e-8)
    return merged

def merge_dare(task_vectors: List[Dict], dropout_rate: float = 0.5) -> Dict:
    merged = {}
    for key in task_vectors[0].keys():
        vectors = torch.stack([tv[key] for tv in task_vectors])
        mask = torch.rand_like(vectors) > dropout_rate
        dropped = vectors * mask
        rescaled = dropped / (1 - dropout_rate)
        merged[key] = rescaled.mean(dim=0)
    return merged

Работа с task vectors

In [ ]:
def get_task_vector(base_model, ft_model) -> Dict:
    task_vector = {}
    with torch.no_grad():
        for name, param in ft_model.named_parameters():
            if name in base_model.state_dict():
                task_vector[name] = (param.data - base_model.state_dict()[name]).cpu()
    return task_vector

def apply_task_vector(base_model, task_vector, alpha: float = 1.0): #применение task vector к базовой модели
    with torch.no_grad():
        for name, param in base_model.named_parameters():
            if name in task_vector:
                tv_gpu = task_vector[name].to(param.device)
                param.data += tv_gpu * alpha

def reset_model_to_base(base_model, base_state_dict):
    with torch.no_grad():
        for name, param in base_model.named_parameters():
            if name in base_state_dict:
                param.data.copy_(base_state_dict[name].to(param.device))

Загрузка базовой модели

In [ ]:
def load_base_model(base_model_name: str): #загрузка базовой модели
    tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        torch_dtype=torch.float32,
        device_map="auto",
        trust_remote_code=True
    )
    model.eval()
    return model, tokenizer

Загрузка CPT модели (базовая + LoRA)

In [ ]:
def load_cpt_model(base_model_name: str, cpt_lora_path: str):
    tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        torch_dtype=torch.float32,
        device_map="auto",
        trust_remote_code=True
    )

    if cpt_lora_path and os.path.exists(cpt_lora_path):
        print(f"   Загрузка CPT LoRA из {cpt_lora_path}")
        model = PeftModel.from_pretrained(model, cpt_lora_path)
        model = model.merge_and_unload()

    model.eval()
    return model, tokenizer

Загрузка FT модели поверх CPT модели

In [ ]:
def load_ft_model_with_cpt_base(base_model_name: str, cpt_model, ft_lora_path: str):
    ft_lora = PeftModel.from_pretrained(cpt_model, ft_lora_path)
    ft_model = ft_lora.merge_and_unload()
    return ft_model

Загрузка CPT модели и вычисление task vectors для FT моделей относительно CPT

In [ ]:
def load_models_with_cpt_sequentially(base_model_name: str, cpt_lora_path: str, ft_lora_paths: Dict):

    cpt_model, tokenizer = load_cpt_model(base_model_name, cpt_lora_path)

    cpt_state_dict = {}
    for name, param in cpt_model.named_parameters():
        cpt_state_dict[name] = param.data.cpu().clone()

    task_vectors = {}

    #Обработка модели
    for name, ft_path in ft_lora_paths.items():
        print(f"   Загрузка {name} LoRA поверх CPT...")

        ft_model = load_ft_model_with_cpt_base(base_model_name, cpt_model, ft_path)
        print(f"   {name} модель загружена. Память GPU: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

        print(f"   Вычисление task vector для {name}...")
        task_vectors[name] = get_task_vector(cpt_model, ft_model)

        print(f"   Выгрузка {name} модели...")
        del ft_model
        gc.collect()
        torch.cuda.empty_cache()
        print(f"   Память GPU после выгрузки: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

    return cpt_model, tokenizer, task_vectors, cpt_state_dict

Функции для расчета метрик

In [ ]:
def calculate_bleu(reference: str, hypothesis: str) -> float: #BLUE
    reference_tokens = word_tokenize(reference.lower())
    hypothesis_tokens = word_tokenize(hypothesis.lower())
    if len(hypothesis_tokens) == 0:
        return 0.0
    smoothing = SmoothingFunction().method1
    bleu = sentence_bleu([reference_tokens], hypothesis_tokens,
                         weights=(0.25, 0.25, 0.25, 0.25),
                         smoothing_function=smoothing)
    return bleu

def calculate_rouge(reference: str, hypothesis: str, scorer) -> Dict: #ROUGE
    scores = scorer.score(reference, hypothesis)
    return {
        'rouge1': scores['rouge1'].fmeasure,
        'rouge2': scores['rouge2'].fmeasure,
        'rougeL': scores['rougeL'].fmeasure
    }

def calculate_semantic_similarity(reference: str, hypothesis: str) -> float: #Семантическое сходство
    global semantic_model
    if semantic_model is None:
        init_semantic_model()

    ref_embedding = semantic_model.encode(reference, convert_to_tensor=True)
    hyp_embedding = semantic_model.encode(hypothesis, convert_to_tensor=True)

    similarity = torch.cosine_similarity(ref_embedding.unsqueeze(0), hyp_embedding.unsqueeze(0))
    return similarity.item()

def extract_key_terms(text: str) -> set:
    key_terms = {
        'neural', 'network', 'learning', 'training', 'model', 'data', 'algorithm',
        'gradient', 'descent', 'backpropagation', 'attention', 'transformer',
        'convolution', 'lstm', 'rnn', 'cnn', 'gpu', 'cpu', 'tensor', 'pytorch',
        'tensorflow', 'keras', 'api', 'classification', 'regression', 'clustering',
        'supervised', 'unsupervised', 'reinforcement', 'q-learning', 'deep', 'ml',
        'ai', 'machine learning', 'deep learning', 'fine-tuning', 'pretrained',
        'overfitting', 'underfitting', 'regularization', 'dropout', 'batch norm',
        'activation', 'relu', 'sigmoid', 'tanh', 'softmax', 'cross-entropy',
        'mse', 'mae', 'optimizer', 'adam', 'sgd', 'rmsprop', 'adagrad'
    }

    words = text.lower().split()
    found_terms = set()

    for word in words:
        word_clean = word.strip('.,!?;:()[]{}"\'-')
        if word_clean in key_terms:
            found_terms.add(word_clean)

    for i in range(len(words) - 1):
        bigram = f"{words[i]} {words[i+1]}".strip('.,!?;:()[]{}"\'-')
        if bigram in key_terms:
            found_terms.add(bigram)

    return found_terms

def calculate_term_precision(reference: str, hypothesis: str) -> float: #Точность терминов
    ref_terms = extract_key_terms(reference)
    hyp_terms = extract_key_terms(hypothesis)

    if len(ref_terms) == 0:
        return 1.0

    overlap = len(ref_terms & hyp_terms)
    return overlap / len(ref_terms)

Функция для оценки логов

In [ ]:
def evaluate_logs(model, tokenizer, test_path, num_samples=20): #
    test_data = []
    with open(test_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                test_data.append(item)

    ok_samples = [item for item in test_data if item["output"] == "OK"]
    anomaly_samples = [item for item in test_data if item["output"] == "Anomaly"]

    num_ok = num_samples // 2
    num_anomaly = num_samples - num_ok

    if len(ok_samples) > num_ok:
        ok_samples = ok_samples[:num_ok]
    if len(anomaly_samples) > num_anomaly:
        anomaly_samples = anomaly_samples[:num_anomaly]

    test_data = ok_samples + anomaly_samples

    system_prompt = "Ты - эксперт по анализу логов. Определи, является ли событие аномалией. Ответь одним словом: Anomaly или OK."

    def predict(input_text):
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": input_text},
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=10,
                do_sample=False,
                temperature=0.1,
                pad_token_id=tokenizer.eos_token_id,
                use_cache=True,
            )

        response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        response_upper = response.upper()

        if "ANOMALY" in response_upper:
            return "Anomaly"
        elif "OK" in response_upper:
            return "OK"
        else:
            return "Unknown"

    y_true = []
    y_pred = []

    for item in tqdm(test_data, desc="Evaluating logs"):
        y_true.append(item["output"])
        pred = predict(item["input"])
        y_pred.append(pred)

    valid_idx = [i for i, p in enumerate(y_pred) if p != "Unknown"]

    if len(valid_idx) == 0:
        return {"accuracy": 0, "precision": 0, "recall": 0, "f1": 0}

    y_true_valid = [y_true[i] for i in valid_idx]
    y_pred_valid = [y_pred[i] for i in valid_idx]

    return {
        "accuracy": accuracy_score(y_true_valid, y_pred_valid),
        "precision": precision_score(y_true_valid, y_pred_valid, pos_label="Anomaly", zero_division=0),
        "recall": recall_score(y_true_valid, y_pred_valid, pos_label="Anomaly", zero_division=0),
        "f1": f1_score(y_true_valid, y_pred_valid, pos_label="Anomaly", zero_division=0)
    }

Функция для оценки на QA датасете

In [ ]:
def evaluate_qa_advanced(model, tokenizer, test_path, num_samples=10):
    test_data = []
    with open(test_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                test_data.append(item)

    test_data = test_data[:num_samples]

    system_prompt = "Ты - эксперт по AI и машинному обучению. Отвечай подробно и развёрнуто, объясняя сложные концепции простым языком. Приводи примеры, если это уместно. Ответ должен быть полезным и информативным."

    rouge_scorer_obj = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    results = {
        'bleu': [],
        'rouge1': [],
        'rouge2': [],
        'rougeL': [],
        'semantic_similarity': [],
        'term_precision': []
    }

    def generate_response(input_text):
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": input_text},
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                temperature=0.1,
                pad_token_id=tokenizer.eos_token_id,
                use_cache=True,
            )

        response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        return response

    for item in tqdm(test_data, desc="Evaluating QA"):
        generated = generate_response(item["input"])

        #BLEU
        bleu_score = calculate_bleu(item["output"], generated)
        results['bleu'].append(bleu_score)

        #ROUGE
        rouge_scores = calculate_rouge(item["output"], generated, rouge_scorer_obj)
        results['rouge1'].append(rouge_scores['rouge1'])
        results['rouge2'].append(rouge_scores['rouge2'])
        results['rougeL'].append(rouge_scores['rougeL'])

        #Semantic Similarity
        semantic_sim = calculate_semantic_similarity(item["output"], generated)
        results['semantic_similarity'].append(semantic_sim)

        #Term Precision
        term_prec = calculate_term_precision(item["output"], generated)
        results['term_precision'].append(term_prec)

    return {
        'bleu': np.mean(results['bleu']),
        'rouge1': np.mean(results['rouge1']),
        'rouge2': np.mean(results['rouge2']),
        'rougeL': np.mean(results['rougeL']),
        'semantic_similarity': np.mean(results['semantic_similarity']),
        'term_precision': np.mean(results['term_precision']),
        'qa_score': np.mean(results['bleu'])
    }

Эксперименты со слиянием

In [ ]:
def run_merge_experiments(base_model, tokenizer, task_vectors, base_state_dict, experiment_name_prefix="cpt"):

    results = {}

    #Linear merge
    for w_logs in [0.3, 0.5, 0.7]:
        w_qa = 1.0 - w_logs
        print(f"\nLinear Merge (logs weight={w_logs}, qa weight={w_qa})")

        reset_model_to_base(base_model, base_state_dict)
        torch.cuda.empty_cache()

        tv_list = [task_vectors["logs"], task_vectors["qa"]]
        weights = [w_logs, w_qa]
        merged_tv = merge_linear(tv_list, weights)
        apply_task_vector(base_model, merged_tv, alpha=1.0)

        logs_metrics = evaluate_logs(base_model, tokenizer, "data/hdfs_test_balanced_1k.jsonl", num_samples=20)
        qa_metrics = evaluate_qa_advanced(base_model, tokenizer, "data/stackexchange_test.jsonl", num_samples=10)

        results[f"linear_logs{w_logs}_qa{w_qa}"] = {
            "logs": logs_metrics,
            "qa": qa_metrics
        }

        print(f"  Логи - Acc: {logs_metrics['accuracy']:.4f}, F1: {logs_metrics['f1']:.4f}")
        print(f"  QA - BLEU: {qa_metrics['bleu']:.4f}, ROUGE-1: {qa_metrics['rouge1']:.4f}")
        print(f"  QA - Semantic Sim: {qa_metrics['semantic_similarity']:.4f}, Term Prec: {qa_metrics['term_precision']:.4f}")

        gc.collect()
        torch.cuda.empty_cache()

    #TIES merge
    for top_k in [0.1, 0.2, 0.3]:
        print(f"\nTIES Merge (top_k={top_k})")

        reset_model_to_base(base_model, base_state_dict)
        torch.cuda.empty_cache()

        tv_list = [task_vectors["logs"], task_vectors["qa"]]
        merged_tv = merge_ties(tv_list, top_k=top_k)
        apply_task_vector(base_model, merged_tv, alpha=1.0)

        logs_metrics = evaluate_logs(base_model, tokenizer, "data/hdfs_test_balanced_1k.jsonl", num_samples=20)
        qa_metrics = evaluate_qa_advanced(base_model, tokenizer, "data/stackexchange_test.jsonl", num_samples=10)

        results[f"ties_topk_{top_k}"] = {
            "logs": logs_metrics,
            "qa": qa_metrics
        }

        print(f"  Логи - Acc: {logs_metrics['accuracy']:.4f}, F1: {logs_metrics['f1']:.4f}")
        print(f"  QA - BLEU: {qa_metrics['bleu']:.4f}, ROUGE-1: {qa_metrics['rouge1']:.4f}")
        print(f"  QA - Semantic Sim: {qa_metrics['semantic_similarity']:.4f}, Term Prec: {qa_metrics['term_precision']:.4f}")

        gc.collect()
        torch.cuda.empty_cache()

    #DARE merge
    for dropout in [0.3, 0.5, 0.7]:
        print(f"\nDARE Merge (dropout={dropout})")

        reset_model_to_base(base_model, base_state_dict)
        torch.cuda.empty_cache()

        tv_list = [task_vectors["logs"], task_vectors["qa"]]
        merged_tv = merge_dare(tv_list, dropout_rate=dropout)
        apply_task_vector(base_model, merged_tv, alpha=1.0)

        logs_metrics = evaluate_logs(base_model, tokenizer, "data/hdfs_test_balanced_1k.jsonl", num_samples=20)
        qa_metrics = evaluate_qa_advanced(base_model, tokenizer, "data/stackexchange_test.jsonl", num_samples=10)

        results[f"dare_dropout_{dropout}"] = {
            "logs": logs_metrics,
            "qa": qa_metrics
        }

        print(f"  Логи - Acc: {logs_metrics['accuracy']:.4f}, F1: {logs_metrics['f1']:.4f}")
        print(f"  QA - BLEU: {qa_metrics['bleu']:.4f}, ROUGE-1: {qa_metrics['rouge1']:.4f}")
        print(f"  QA - Semantic Sim: {qa_metrics['semantic_similarity']:.4f}, Term Prec: {qa_metrics['term_precision']:.4f}")

        gc.collect()
        torch.cuda.empty_cache()

    #Sequential merge
    print("\n    Sequential Merge")

    reset_model_to_base(base_model, base_state_dict)
    torch.cuda.empty_cache()

    tv_list = [task_vectors["logs"], task_vectors["qa"]]
    merged_tv_step1 = merge_linear(tv_list, [0.5, 0.5])
    apply_task_vector(base_model, merged_tv_step1, alpha=0.5)

    intermediate_state_dict = {}
    for name, param in base_model.named_parameters():
        intermediate_state_dict[name] = param.data.cpu().clone()

    reset_model_to_base(base_model, base_state_dict)

    seq_tv = {}
    for name in base_state_dict.keys():
        if name in intermediate_state_dict:
            seq_tv[name] = (intermediate_state_dict[name] - base_state_dict[name])

    apply_task_vector(base_model, seq_tv, alpha=0.7)

    logs_metrics = evaluate_logs(base_model, tokenizer, "data/hdfs_test_balanced_1k.jsonl", num_samples=20)
    qa_metrics = evaluate_qa_advanced(base_model, tokenizer, "data/stackexchange_test.jsonl", num_samples=10)

    results["sequential"] = {
        "logs": logs_metrics,
        "qa": qa_metrics
    }

    print(f"  Логи - Acc: {logs_metrics['accuracy']:.4f}, F1: {logs_metrics['f1']:.4f}")
    print(f"  QA - BLEU: {qa_metrics['bleu']:.4f}, ROUGE-1: {qa_metrics['rouge1']:.4f}")
    print(f"  QA - Semantic Sim: {qa_metrics['semantic_similarity']:.4f}, Term Prec: {qa_metrics['term_precision']:.4f}")

    #Weight averaging
    print("\n    Weight Averaging")

    reset_model_to_base(base_model, base_state_dict)

    for name in base_model.state_dict().keys():
        if name in task_vectors["logs"] and name in task_vectors["qa"]:
            weight_logs = base_state_dict[name] + task_vectors["logs"][name]
            weight_qa = base_state_dict[name] + task_vectors["qa"][name]
            base_model.state_dict()[name].copy_(((weight_logs + weight_qa) / 2).to(base_model.device))

    logs_metrics = evaluate_logs(base_model, tokenizer, "data/hdfs_test_balanced_1k.jsonl", num_samples=20)
    qa_metrics = evaluate_qa_advanced(base_model, tokenizer, "data/stackexchange_test.jsonl", num_samples=10)

    results["weight_averaging"] = {
        "logs": logs_metrics,
        "qa": qa_metrics
    }

    print(f"  Логи - Acc: {logs_metrics['accuracy']:.4f}, F1: {logs_metrics['f1']:.4f}")
    print(f"  QA - BLEU: {qa_metrics['bleu']:.4f}, ROUGE-1: {qa_metrics['rouge1']:.4f}")
    print(f"  QA - Semantic Sim: {qa_metrics['semantic_similarity']:.4f}, Term Prec: {qa_metrics['term_precision']:.4f}")

    return results

Слияние моделей

In [ ]:
def main():
    init_semantic_model()

    base_model_name = "unsloth/Qwen3-4B-Instruct-2507"
    cpt_lora_path = "qwen_cpt_results"
    ft_lora_paths = {
        "logs": "qwen_hdfs_results",
        "qa": "qwen_qa_results"
    }

    #загрузка CPT модели и вычисление task vectors для FT моделей относительно CPT
    cpt_model, tokenizer, task_vectors, cpt_state_dict = load_models_with_cpt_sequentially(
        base_model_name, cpt_lora_path, ft_lora_paths
    )

    #ОЦЕНКА CPT МОДЕЛИ
    print("ОЦЕНКА CPT МОДЕЛИ")

    cpt_logs_metrics = evaluate_logs(cpt_model, tokenizer, "data/hdfs_test_balanced_1k.jsonl", num_samples=20)
    cpt_qa_metrics = evaluate_qa_advanced(cpt_model, tokenizer, "data/stackexchange_test.jsonl", num_samples=10)

    print(f"CPT модель - Логи: Acc={cpt_logs_metrics['accuracy']:.4f}, F1={cpt_logs_metrics['f1']:.4f}")
    print(f"CPT модель - QA: BLEU={cpt_qa_metrics['bleu']:.4f}, ROUGE-1={cpt_qa_metrics['rouge1']:.4f}")
    print(f"CPT модель - Semantic Similarity: {cpt_qa_metrics['semantic_similarity']:.4f}")
    print(f"CPT модель - Term Precision: {cpt_qa_metrics['term_precision']:.4f}")

    #запуск экспериментов со слиянием
    results = run_merge_experiments(cpt_model, tokenizer, task_vectors, cpt_state_dict, "cpt")

    print("\nCPT модель (базовая):")
    print(f"  Логи: Acc={cpt_logs_metrics['accuracy']:.4f}, F1={cpt_logs_metrics['f1']:.4f}")
    print(f"  QA: BLEU={cpt_qa_metrics['bleu']:.4f}, ROUGE-1={cpt_qa_metrics['rouge1']:.4f}")
    print(f"  QA: Semantic Similarity={cpt_qa_metrics['semantic_similarity']:.4f}, Term Precision={cpt_qa_metrics['term_precision']:.4f}")

    print("\nРезультаты слияний:")
    for method, metrics in results.items():
        print(f"\n{method}:")
        print(f"  Логи - Acc: {metrics['logs']['accuracy']:.4f}, F1: {metrics['logs']['f1']:.4f}")
        print(f"  QA - BLEU: {metrics['qa']['bleu']:.4f}, ROUGE-1: {metrics['qa']['rouge1']:.4f}")
        print(f"  QA - Semantic Sim: {metrics['qa']['semantic_similarity']:.4f}, Term Prec: {metrics['qa']['term_precision']:.4f}")

    #нахождение лушчих методов
    print("ЛУЧШИЕ МЕТОДЫ ПО РАЗНЫМ МЕТРИКАМ")

    best_for_logs = max(results.items(), key=lambda x: x[1]['logs']['f1'])
    best_for_bleu = max(results.items(), key=lambda x: x[1]['qa']['bleu'])
    best_for_semantic = max(results.items(), key=lambda x: x[1]['qa']['semantic_similarity'])
    best_for_terms = max(results.items(), key=lambda x: x[1]['qa']['term_precision'])

    print(f"\nЛучший метод для логов (по F1): {best_for_logs[0]}")
    print(f"  Логи F1: {best_for_logs[1]['logs']['f1']:.4f}")
    print(f"  QA BLEU: {best_for_logs[1]['qa']['bleu']:.4f}")
    print(f"  QA Semantic: {best_for_logs[1]['qa']['semantic_similarity']:.4f}")

    print(f"\nЛучший метод для QA (по Semantic Similarity): {best_for_semantic[0]}")
    print(f"  QA Semantic: {best_for_semantic[1]['qa']['semantic_similarity']:.4f}")
    print(f"  QA Term Precision: {best_for_semantic[1]['qa']['term_precision']:.4f}")
    print(f"  Логи F1: {best_for_semantic[1]['logs']['f1']:.4f}")

    print(f"\nЛучший метод для QA (по Term Precision): {best_for_terms[0]}")
    print(f"  QA Term Precision: {best_for_terms[1]['qa']['term_precision']:.4f}")
    print(f"  QA Semantic: {best_for_terms[1]['qa']['semantic_similarity']:.4f}")
    print(f"  Логи F1: {best_for_terms[1]['logs']['f1']:.4f}")

    #СРАВНЕНИЕ С CPT МОДЕЛЬЮ
    print("СРАВНЕНИЕ С CPT МОДЕЛЬЮ")

    if best_for_logs[1]['logs']['f1'] > cpt_logs_metrics['f1']:
        print(f"Лучший метод ({best_for_logs[0]}) улучшает логи: +{best_for_logs[1]['logs']['f1'] - cpt_logs_metrics['f1']:.4f} F1")
    else:
        print(f"Лучший метод ({best_for_logs[0]}) ухудшает логи: {best_for_logs[1]['logs']['f1'] - cpt_logs_metrics['f1']:.4f} F1")

    if best_for_semantic[1]['qa']['semantic_similarity'] > cpt_qa_metrics['semantic_similarity']:
        print(f"Лучший метод ({best_for_semantic[0]}) улучшает семантику: +{best_for_semantic[1]['qa']['semantic_similarity'] - cpt_qa_metrics['semantic_similarity']:.4f}")
    else:
        print(f"Лучший метод ({best_for_semantic[0]}) ухудшает семантику: {best_for_semantic[1]['qa']['semantic_similarity'] - cpt_qa_metrics['semantic_similarity']:.4f}")

    if best_for_terms[1]['qa']['term_precision'] > cpt_qa_metrics['term_precision']:
        print(f"Лучший метод ({best_for_terms[0]}) улучшает точность терминов: +{best_for_terms[1]['qa']['term_precision'] - cpt_qa_metrics['term_precision']:.4f}")
    else:
        print(f"Лучший метод ({best_for_terms[0]}) ухудшает точность терминов: {best_for_terms[1]['qa']['term_precision'] - cpt_qa_metrics['term_precision']:.4f}")

    del cpt_model
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
if __name__ == "__main__":
    main()

Вопрос: "А почему этот код вывел все метрики по всем методам слияния одинаковыми?"  

Проблема в том, что task vectors для логов и QA оказались нулевыми или очень маленькими. Это происходит из-за того, как мы загружаем FT модели поверх CPT.  
Иначе говоря дело в том, что LoRA адаптер для FT модели хранит только дельты относительно исходной базовой модели, но мы применяем его к CPT модели, у которой веса уже изменены. Это приводит к непредсказуемым результатам.  

Нужно загружать FT модели относительно исходной базовой модели, а затем вычислять task vectors относительно CPT

Корректный расчет task vectors FT моделей относительно CPT

In [ ]:
def load_models_with_cpt_sequentially_fixed(base_model_name: str, cpt_lora_path: str, ft_lora_paths: Dict):

    #загрузка исходной базовой модели
    base_model, tokenizer = load_base_model(base_model_name)

    #сохранение весов базовой модели
    base_state_dict = {}
    for name, param in base_model.named_parameters():
        base_state_dict[name] = param.data.cpu().clone()

    #загрузка CPT модели
    cpt_model, _ = load_cpt_model(base_model_name, cpt_lora_path)

    #сохраняем веса CPT модели для сброса
    cpt_state_dict = {}
    for name, param in cpt_model.named_parameters():
        cpt_state_dict[name] = param.data.cpu().clone()

    task_vectors = {}

    #для каждой FT модели
    for name, ft_path in ft_lora_paths.items():

        #обработка модели

        #загрузка FT модели от исходной базы
        ft_model, _ = load_base_model(base_model_name)
        ft_lora = PeftModel.from_pretrained(ft_model, ft_path)
        ft_model = ft_lora.merge_and_unload()

        #вычисление delta = W_ft - W_base
        delta_state_dict = {}
        for name_param, param in ft_model.named_parameters():
            if name_param in base_state_dict:
                delta_state_dict[name_param] = (param.data.cpu() - base_state_dict[name_param])

        #task vector относительно CPT
        task_vectors[name] = delta_state_dict

        #проверка нормы
        total_norm = 0.0
        for v in task_vectors[name].values():
            total_norm += torch.norm(v).item() ** 2
        total_norm = np.sqrt(total_norm)
        print(f"   Норма task vector для {name}: {total_norm:.6f}")

        if total_norm < 1e-6:
            print(f"Task vector для {name} почти нулевой!")

        #FT модель
        del ft_model, ft_lora
        gc.collect()
        torch.cuda.empty_cache()

    del base_model
    gc.collect()
    torch.cuda.empty_cache()

    return cpt_model, tokenizer, task_vectors, cpt_state_dict

def load_models_all_from_base(base_model_name: str, cpt_path: str, ft_paths: Dict):

    #загрузка исходной базовой модели
    base_model, tokenizer = load_base_model(base_model_name)

    base_state_dict = {}
    for name, param in base_model.named_parameters():
        base_state_dict[name] = param.data.cpu().clone()

    #загрузка CPT модели
    cpt_model = load_ft_model_with_cpt_base(base_model_name, base_model, cpt_path)

    #вычисление task vectors для CPT
    cpt_task_vector = get_task_vector(base_model, cpt_model)

    #CPT модель
    del cpt_model
    gc.collect()
    torch.cuda.empty_cache()

    #FT модели
    task_vectors = {"cpt": cpt_task_vector}
    for name, ft_path in ft_paths.items():
        ft_model, _ = load_base_model(base_model_name)  #загрузка базовой
        ft_lora = PeftModel.from_pretrained(ft_model, ft_path)
        ft_model = ft_lora.merge_and_unload()

        task_vectors[name] = get_task_vector(base_model, ft_model)

        del ft_model
        gc.collect()
        torch.cuda.empty_cache()

    return base_model, tokenizer, task_vectors, base_state_dict

Слияние моделей

In [ ]:
def main():
    init_semantic_model()

    base_model_name = "unsloth/Qwen3-4B-Instruct-2507"
    cpt_lora_path = "qwen_cpt_results"
    ft_lora_paths = {
        "logs": "qwen_hdfs_results",
        "qa": "qwen_qa_results"
    }

    #загрузка CPT модели и вычисление task vectors для FT моделей относительно CPT
    cpt_model, tokenizer, task_vectors, cpt_state_dict = load_models_with_cpt_sequentially_fixed(
        base_model_name, cpt_lora_path, ft_lora_paths
    )

    #ОЦЕНКА CPT МОДЕЛИ
    print("ОЦЕНКА CPT МОДЕЛИ")

    cpt_logs_metrics = evaluate_logs(cpt_model, tokenizer, "data/hdfs_test_balanced_1k.jsonl", num_samples=20)
    cpt_qa_metrics = evaluate_qa_advanced(cpt_model, tokenizer, "data/stackexchange_test.jsonl", num_samples=10)

    print(f"CPT модель - Логи: Acc={cpt_logs_metrics['accuracy']:.4f}, F1={cpt_logs_metrics['f1']:.4f}")
    print(f"CPT модель - QA: BLEU={cpt_qa_metrics['bleu']:.4f}, ROUGE-1={cpt_qa_metrics['rouge1']:.4f}")
    print(f"CPT модель - Semantic Similarity: {cpt_qa_metrics['semantic_similarity']:.4f}")
    print(f"CPT модель - Term Precision: {cpt_qa_metrics['term_precision']:.4f}")

    #запуск экспериментов со слиянием
    results = run_merge_experiments(cpt_model, tokenizer, task_vectors, cpt_state_dict, "cpt")

    print("\nCPT модель (базовая):")
    print(f"  Логи: Acc={cpt_logs_metrics['accuracy']:.4f}, F1={cpt_logs_metrics['f1']:.4f}")
    print(f"  QA: BLEU={cpt_qa_metrics['bleu']:.4f}, ROUGE-1={cpt_qa_metrics['rouge1']:.4f}")
    print(f"  QA: Semantic Similarity={cpt_qa_metrics['semantic_similarity']:.4f}, Term Precision={cpt_qa_metrics['term_precision']:.4f}")

    print("\nРезультаты слияний:")
    for method, metrics in results.items():
        print(f"\n{method}:")
        print(f"  Логи - Acc: {metrics['logs']['accuracy']:.4f}, F1: {metrics['logs']['f1']:.4f}")
        print(f"  QA - BLEU: {metrics['qa']['bleu']:.4f}, ROUGE-1: {metrics['qa']['rouge1']:.4f}")
        print(f"  QA - Semantic Sim: {metrics['qa']['semantic_similarity']:.4f}, Term Prec: {metrics['qa']['term_precision']:.4f}")

    #нахождение лушчих методов
    print("ЛУЧШИЕ МЕТОДЫ ПО РАЗНЫМ МЕТРИКАМ")

    best_for_logs = max(results.items(), key=lambda x: x[1]['logs']['f1'])
    best_for_bleu = max(results.items(), key=lambda x: x[1]['qa']['bleu'])
    best_for_semantic = max(results.items(), key=lambda x: x[1]['qa']['semantic_similarity'])
    best_for_terms = max(results.items(), key=lambda x: x[1]['qa']['term_precision'])

    print(f"\nЛучший метод для логов (по F1): {best_for_logs[0]}")
    print(f"  Логи F1: {best_for_logs[1]['logs']['f1']:.4f}")
    print(f"  QA BLEU: {best_for_logs[1]['qa']['bleu']:.4f}")
    print(f"  QA Semantic: {best_for_logs[1]['qa']['semantic_similarity']:.4f}")

    print(f"\nЛучший метод для QA (по Semantic Similarity): {best_for_semantic[0]}")
    print(f"  QA Semantic: {best_for_semantic[1]['qa']['semantic_similarity']:.4f}")
    print(f"  QA Term Precision: {best_for_semantic[1]['qa']['term_precision']:.4f}")
    print(f"  Логи F1: {best_for_semantic[1]['logs']['f1']:.4f}")

    print(f"\nЛучший метод для QA (по Term Precision): {best_for_terms[0]}")
    print(f"  QA Term Precision: {best_for_terms[1]['qa']['term_precision']:.4f}")
    print(f"  QA Semantic: {best_for_terms[1]['qa']['semantic_similarity']:.4f}")
    print(f"  Логи F1: {best_for_terms[1]['logs']['f1']:.4f}")

    #СРАВНЕНИЕ С CPT МОДЕЛЬЮ
    print("СРАВНЕНИЕ С CPT МОДЕЛЬЮ")

    if best_for_logs[1]['logs']['f1'] > cpt_logs_metrics['f1']:
        print(f"Лучший метод ({best_for_logs[0]}) улучшает логи: +{best_for_logs[1]['logs']['f1'] - cpt_logs_metrics['f1']:.4f} F1")
    else:
        print(f"Лучший метод ({best_for_logs[0]}) ухудшает логи: {best_for_logs[1]['logs']['f1'] - cpt_logs_metrics['f1']:.4f} F1")

    if best_for_semantic[1]['qa']['semantic_similarity'] > cpt_qa_metrics['semantic_similarity']:
        print(f"Лучший метод ({best_for_semantic[0]}) улучшает семантику: +{best_for_semantic[1]['qa']['semantic_similarity'] - cpt_qa_metrics['semantic_similarity']:.4f}")
    else:
        print(f"Лучший метод ({best_for_semantic[0]}) ухудшает семантику: {best_for_semantic[1]['qa']['semantic_similarity'] - cpt_qa_metrics['semantic_similarity']:.4f}")

    if best_for_terms[1]['qa']['term_precision'] > cpt_qa_metrics['term_precision']:
        print(f"Лучший метод ({best_for_terms[0]}) улучшает точность терминов: +{best_for_terms[1]['qa']['term_precision'] - cpt_qa_metrics['term_precision']:.4f}")
    else:
        print(f"Лучший метод ({best_for_terms[0]}) ухудшает точность терминов: {best_for_terms[1]['qa']['term_precision'] - cpt_qa_metrics['term_precision']:.4f}")

    del cpt_model
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
if __name__ == "__main__":
    main()